# 09 — Test-set inference for every saved run

For each `runs/<run_id>/result.json`, re-evaluate the same config on the **test** set
(originally the ImageNet validation split at `/home/pf4636/imagenet2/val`) and write a
fresh result JSON to `runs/final_runs/<run_id>/`.

- pytorch backend → load weights → eval on test loader
- torchao_cpu_ptq → re-calibrate on train loader → eval on test loader
- tensorrt → reuse existing engine in `engines/` → trt_evaluate on test loader

In [1]:
import sys, json, time
from pathlib import Path
from dataclasses import replace

SRC = Path("..").resolve() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import ExperimentConfig, set_seed
from data import build_imagenet_dataset, build_runner_loaders
from model import get_model
from precision import apply_precision
from quant_ptq_cpu import quantize_int8_x86_pt2e
from eval import evaluate
from utils import iter_result_jsons, read_json, write_json
from runner import _get_trt_paths

TEST_ROOT  = "/home/pf4636/imagenet2"
RUNS_IN    = Path("../runs/val_infer").resolve()
RUNS_OUT   = Path("../runs/final_runs").resolve()
RUNS_OUT.mkdir(parents=True, exist_ok=True)
print(f"reading from: {RUNS_IN}")
print(f"writing to  : {RUNS_OUT}")

reading from: /home/pf4636/code/resnet/quantized_resnets/runs/val_infer
writing to  : /home/pf4636/code/resnet/quantized_resnets/runs/final_runs


In [2]:
def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_ROOT)
    dataset = build_imagenet_dataset(test_cfg, split="val")
    pin_memory = str(cfg.device).startswith("cuda")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        drop_last=True,
    )

def run_on_test(cfg: ExperimentConfig, criterion=None):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    test_loader = build_test_loader(cfg)
    t0 = time.perf_counter()

    if cfg.backend == "pytorch":
        model = apply_precision(get_model(cfg), cfg)
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "torchao_cpu_ptq":
        train_loader, _ = build_runner_loaders(cfg)
        model = quantize_int8_x86_pt2e(
            get_model(cfg), train_loader,
            calib_num_batches=cfg.cpu_calib_num_batches,
        )
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "tensorrt":
        from trt_infer import trt_evaluate
        from onnx_exporter import ONNXExporter
        from trt_builder import build_engine

        onnx_path, engine_path, _ = _get_trt_paths(cfg)
        if not onnx_path.exists():
            ONNXExporter(get_model(cfg), cfg.device, onnx_path).export_model(
                opset_version=cfg.trt_opset if cfg.trt_opset > 1 else 17,
                dynamic_batch=True,
                dummy_input_shape=(1, 3, 224, 224),
            )
        if not engine_path.exists():
            build_engine(
                onnx_path=onnx_path,
                engine_path=engine_path,
                precision=cfg.model_precision,
                batch_size=cfg.batch_size,
                workspace_mb=cfg.trt_workspace_mb,
            )
        tracker = trt_evaluate(engine_path, cfg, test_loader, criterion)

    else:
        raise ValueError(f"Unknown backend: {cfg.backend}")

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "error": None,
        "total_eval_time_sec": round(time.perf_counter() - t0, 3),
    }
    return payload, tracker

In [3]:
def cfg_from_payload(payload: dict) -> ExperimentConfig:
    raw = payload["config"]
    fields = {f for f in ExperimentConfig.__dataclass_fields__}
    return ExperimentConfig(**{k: v for k, v in raw.items() if k in fields})

configs = []
for p in iter_result_jsons(RUNS_IN):
    if RUNS_OUT in p.parents:
        continue
    payload = read_json(p)
    if payload.get("status") != "ok":
        continue
    configs.append(cfg_from_payload(payload))

print(f"found {len(configs)} runs to re-evaluate on test set")
for c in configs:
    print("  ", c.run_id())

found 32 runs to re-evaluate on test set
   resnet18_pytorch_fp16_in1b_cuda_bs1
   resnet18_pytorch_fp16_in2b_cuda_bs1
   resnet18_pytorch_fp16_in4b_cuda_bs1
   resnet18_pytorch_fp16_in8b_cuda_bs1
   resnet18_pytorch_fp32_in1b_cuda_bs1
   resnet18_pytorch_fp32_in2b_cuda_bs1
   resnet18_pytorch_fp32_in4b_cuda_bs1
   resnet18_pytorch_fp32_in8b_cuda_bs1
   resnet18_tensorrt_fp16_in1b_cuda_bs1
   resnet18_tensorrt_fp16_in2b_cuda_bs1
   resnet18_tensorrt_fp16_in4b_cuda_bs1
   resnet18_tensorrt_fp16_in8b_cuda_bs1
   resnet18_tensorrt_fp32_in1b_cuda_bs1
   resnet18_tensorrt_fp32_in2b_cuda_bs1
   resnet18_tensorrt_fp32_in4b_cuda_bs1
   resnet18_tensorrt_fp32_in8b_cuda_bs1
   resnet18_tensorrt_fp8_in1b_cuda_bs1
   resnet18_tensorrt_fp8_in2b_cuda_bs1
   resnet18_tensorrt_fp8_in4b_cuda_bs1
   resnet18_tensorrt_fp8_in8b_cuda_bs1
   resnet18_tensorrt_int4_in1b_cuda_bs1
   resnet18_tensorrt_int4_in2b_cuda_bs1
   resnet18_tensorrt_int4_in4b_cuda_bs1
   resnet18_tensorrt_int4_in8b_cuda_bs1
   resnet18

In [4]:
SKIP_EXISTING = True

results_summary = []
for i, cfg in enumerate(configs, 1):
    out_path = RUNS_OUT / cfg.run_id() / "result.json"
    print(f"\n[{i}/{len(configs)}] {cfg.run_id()}")

    if SKIP_EXISTING and out_path.exists():
        print(f"  already exists, skipping: {out_path}")
        results_summary.append(read_json(out_path))
        continue

    try:
        payload, _ = run_on_test(cfg)
        write_json(out_path, payload)
        print(f"  saved: {out_path}")
        r = payload["results"]
        print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f} ms")
        results_summary.append(payload)
    except Exception as e:
        err_payload = {
            "status": "error",
            "run_id": cfg.run_id(),
            "config": cfg.to_dict(),
            "error": f"{type(e).__name__}: {e}",
        }
        write_json(out_path, err_payload)
        print(f"  FAILED: {e}")
        results_summary.append(err_payload)


[1/32] resnet18_pytorch_fp16_in1b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantized_resnets/runs/final_runs/resnet18_pytorch_fp16_in1b_cuda_bs1/result.json

[2/32] resnet18_pytorch_fp16_in2b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantized_resnets/runs/final_runs/resnet18_pytorch_fp16_in2b_cuda_bs1/result.json

[3/32] resnet18_pytorch_fp16_in4b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantized_resnets/runs/final_runs/resnet18_pytorch_fp16_in4b_cuda_bs1/result.json

[4/32] resnet18_pytorch_fp16_in8b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantized_resnets/runs/final_runs/resnet18_pytorch_fp16_in8b_cuda_bs1/result.json

[5/32] resnet18_pytorch_fp32_in1b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantized_resnets/runs/final_runs/resnet18_pytorch_fp32_in1b_cuda_bs1/result.json

[6/32] resnet18_pytorch_fp32_in2b_cuda_bs1
  already exists, skipping: /home/pf4636/code/resnet/quantize

In [5]:
import pandas as pd
from utils import flatten_runs, load_runs

test_runs = load_runs(RUNS_OUT, status="ok")
df = pd.DataFrame(flatten_runs(test_runs))
cols = ["run_id", "cfg.backend", "cfg.model_precision", "cfg.input_quant_bits",
        "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.infer_ms_std"]
df[[c for c in cols if c in df.columns]].sort_values(
    ["cfg.backend", "cfg.input_quant_bits", "cfg.model_precision"]
).reset_index(drop=True)

,run_id,cfg.backend,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.infer_ms_std
0,resnet18_pytorch_fp16_in1b_cuda_bs1,pytorch,fp16,1,31.914894,63.617021,3.041370,1.503594
1,resnet18_pytorch_fp32_in1b_cuda_bs1,pytorch,fp32,1,31.489362,63.617021,3.053116,1.303294
2,resnet18_pytorch_fp16_in2b_cuda_bs1,pytorch,fp16,2,51.914894,81.063830,2.943886,1.567814
3,resnet18_pytorch_fp32_in2b_cuda_bs1,pytorch,fp32,2,51.914894,81.063830,2.840510,1.230063
4,resnet18_pytorch_fp16_in4b_cuda_bs1,pytorch,fp16,4,80.851064,96.595745,2.955571,1.482245
5,resnet18_pytorch_fp32_in4b_cuda_bs1,pytorch,fp32,4,80.851064,96.595745,2.713301,1.138876
6,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,fp16,8,80.638298,97.234043,2.909420,1.458176
7,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,fp32,8,80.638298,97.234043,2.911906,1.322866
8,resnet18_tensorrt_fp16_in1b_cuda_bs1,tensorrt,fp16,1,31.914894,63.404255,0.463038,0.171537
9,resnet18_tensorrt_fp32_in1b_cuda_bs1,tensorrt,fp32,1,31.702128,63.617021,0.931417,0.129937


In [6]:
# 12 trials for measurement uncertainty on resnet18_tensorrt_fp32_in8b_cuda_bs1
N_TRIALS = 12
target_run_id = "resnet18_tensorrt_fp32_in8b_cuda_bs1"

trial_cfg = next(c for c in configs if c.run_id() == target_run_id)
trial_dir = RUNS_OUT / target_run_id
trial_dir.mkdir(parents=True, exist_ok=True)

trial_payloads = []
for i in range(1, N_TRIALS + 1):
    out_path = trial_dir / f"result_{i:03d}.json"
    if out_path.exists():
        print(f"[{i:02d}/{N_TRIALS}] exists, skipping {out_path.name}")
        trial_payloads.append(read_json(out_path))
        continue
    payload, _ = run_on_test(trial_cfg)
    write_json(out_path, payload)
    r = payload["results"]
    print(f"[{i:02d}/{N_TRIALS}] infer_ms_avg={r['infer_ms_avg']:.4f}  top1={r['top1_acc']:.2f}%  saved {out_path.name}")
    trial_payloads.append(payload)

[data] Filtered to 5000 samples across 100 classes.
[trt_builder] Building engine | precision=fp32 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/resnet18_tensorrt_fp32_in8b_cuda_bs1.engine (48.7 MB)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/resnet18_tensorrt_fp32_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/500]  Top-1: 90.00%  Top-5: 100.00%  Infer: 1.00 ms/batch
  [50/500]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.99 ms/batch
  [60/500]  Top-1: 96.67%  Top-5: 100.00%  Infer: 0.98 ms/batch
  [70/500]  Top-1: 92.50%  Top-5: 100.00%  Infer: 1.00 ms/batch
  [80/500]  Top-1: 90.00%  Top-5: 100.00%  Infer: 1.00 ms/batch
  [90/500]  Top-1: 91.67%  Top-5: 100.00%  Infer: 1.00 ms/batch
  [100/500]  Top-1: 90.00%  Top

In [7]:
import numpy as np

infer_ms = np.array([p["results"]["infer_ms_avg"] for p in trial_payloads])
n = len(infer_ms)
mean = infer_ms.mean()
std = infer_ms.std(ddof=1)
ci_half = 1.96 * std / np.sqrt(n)

print(f"Config: {target_run_id}")
print(f"N trials: {n}")
print(f"infer_ms_avg per trial: {np.round(infer_ms, 4).tolist()}")
print(f"mean   = {mean:.4f} ms")
print(f"std    = {std:.4f} ms   (sample std, ddof=1)")
print(f"min    = {infer_ms.min():.4f} ms")
print(f"max    = {infer_ms.max():.4f} ms")
print(f"95% CI = [{mean - ci_half:.4f}, {mean + ci_half:.4f}] ms")

Config: resnet18_tensorrt_fp32_in8b_cuda_bs1
N trials: 12
infer_ms_avg per trial: [1.0095, 0.975, 0.9565, 0.9545, 0.9377, 0.9698, 0.9327, 0.9425, 0.9413, 0.9398, 0.9267, 0.9387]
mean   = 0.9521 ms
std    = 0.0232 ms   (sample std, ddof=1)
min    = 0.9267 ms
max    = 1.0095 ms
95% CI = [0.9389, 0.9652] ms
